# Notebook 03_1 — Predictive Performance (Text + Image Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Women's Shoes (Size 8)

---

Same as 03_2 but uses **multimodal embeddings** (RoBERTa + BEiT + SAINT).  
Adds `x_emb` as a fourth feature specification on top of x, x_pca, x_sim.

Model specifications compared:
- OLS / Boosting — Tabular only
- OLS / Boosting — Tabular + PCA
- OLS / Boosting — Tabular + Similarities
- OLS / Boosting — Tabular + Embeddings (multimodal)
- Deep Time Independent
- Deep Time Dependent

## ① Mount Drive

In [ ]:
# ── Bucket-scoped output config (added by refactor) ─────────────────────
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError('Cannot find project root')

BRAND_BUCKET = 'women_top1_dream_pairs'   # ← change per bucket run
PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data' / BRAND_BUCKET
ANALYSIS_DIR = DATA_ROOT / 'analysis_datasets'
KEY_DIR      = DATA_ROOT / 'keys'
DATA_DIR     = DATA_ROOT / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
PRED_DIR     = DATA_ROOT / 'predictions'
OUTPUT_DIR   = DATA_ROOT / 'output' / 'predictive_performance_txt_img'
IMG_DIR      = PROJECT_ROOT / 'data' / 'images'   # shared (ASIN-unique)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'BRAND_BUCKET : {BRAND_BUCKET}')
print(f'OUTPUT_DIR   : {OUTPUT_DIR}')

## ② Set Working Directory

## ③ Imports

In [6]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the **multimodal** embeddings dataset from notebook 01_1 (txt_only=False).
Key difference from 03_2: this dataset includes both text AND image embeddings.

In [7]:
txt_only = False
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(ANALYSIS_DIR / f"{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(ANALYSIS_DIR / f"{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (30876, 333)
Val shape:   (30888, 333)
Dummy subcat names: ['Other', 'Fashion Sneakers', 'Ballet & Dance', 'Walking', 'Flats', 'Mules & Clogs', 'Ankle & Bootie', 'Pumps', 'Road Running', 'Loafers & Slip-Ons', 'Heeled Sandals']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=333)

## ⑤ Sanity Check — Row Counts

In [8]:
print(f"Val rows:   {len(df_full_val)}")
print(f"Train rows: {len(df_full_train)}")

Val rows:   30888
Train rows: 30876


## ⑥ Define Controls and Feature Specifications

Same controls as 03_2 plus `controls_emb` — all 256 multimodal embedding dimensions.

In [9]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]

# Key difference from 03_2 — multimodal emb columns
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         23
Total controls:         28
Embedding columns:      256


## ⑦ Initialize Results DataFrames

In [10]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑧ Deep Model R² — Level

In [11]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):   {r2_ml_l}")
    print(f"  R2 Treatment (time independent):  {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):                {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):              {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):   0.8223
  R2 Treatment (time independent):  0.7756
  R2 Outcome (lag1):                0.9021
  R2 Treatment (lag1):              0.7796

Evaluation for Test set
  R2 Outcome (time independent):   0.7163
  R2 Treatment (time independent):  0.639
  R2 Outcome (lag1):                0.8913
  R2 Treatment (lag1):              0.6448



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.8223,0.7163,0.7756,0.6390
Deep Time Dependent,0.9021,0.8913,0.7796,0.6448


## ⑨ Deep Model R² — Diff

In [12]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):   {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent):  {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):                {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):              {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):   0.11042887
  R2 Treatment diff (time independent):  -0.0195138
  R2 Outcome diff (lag1):                0.06963254
  R2 Treatment diff (lag1):              -0.00936407

Evaluation for Test set
  R2 Outcome diff (time independent):   0.07314007
  R2 Treatment diff (time independent):  -0.01742634
  R2 Outcome diff (lag1):                0.06388012
  R2 Treatment diff (lag1):              -0.00923879



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.110429,0.07314,-0.019514,-0.017426
Deep Time Dependent,0.069633,0.06388,-0.009364,-0.009239


## ⑩ Build Feature Matrices

Key difference from 03_2: adds `x_train_emb` and `x_test_emb`
using all 256 multimodal embedding dimensions.

In [13]:
# Train set
y_train      = df_full_train[outcome].squeeze()
d_train      = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])
x_train_emb = sm.add_constant(df_full_train[additional_controls + controls_emb])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])
x_test_emb = sm.add_constant(df_full_val[additional_controls + controls_emb])

# Rename columns for LightGBM
for df in [x_train, x_test, x_train_pca, x_test_pca,
           x_train_sim, x_test_sim, x_train_emb, x_test_emb]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")
print(f"x_train_emb shape: {x_train_emb.shape}")

x_train shape:     (30876, 29)
x_train_pca shape: (30876, 34)
x_train_sim shape: (30876, 34)
x_train_emb shape: (30876, 285)


## ⑪ Build Dict Structures

In [14]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca,
    "x_sim": x_train_sim, "x_emb": x_train_emb,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca,
    "x_sim": x_test_sim, "x_emb": x_test_emb,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑫ Tabular Models — Level

Four feature specifications including the new x_emb (Tabular + full embeddings).

In [15]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS
  R2 Outcome train/test: 0.1982 / 0.1688
  R2 Treatment train/test: 0.2404 / 0.1926
  Boosting
  R2 Outcome train/test: 0.5928 / 0.5222
  R2 Treatment train/test: 0.5621 / 0.4045

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.6590 / 0.5755
  R2 Treatment train/test: 0.6190 / 0.5102
  Boosting
  R2 Outcome train/test: 0.9022 / 0.7717
  R2 Treatment train/test: 0.8797 / 0.6908

 Feature specification: x_sim
  OLS
  R2 Outcome train/test: 0.6767 / 0.5878
  R2 Treatment train/test: 0.6284 / 0.5142
  Boosting
  R2 Outcome train/test: 0.8829 / 0.7568
  R2 Treatment train/test: 0.8425 / 0.6518

 Feature specification: x_emb
  OLS
  R2 Outcome train/test: 0.8307 / 0.7090
  R2 Treatment train/test: 0.7897 / 0.6385
  Boosting
  R2 Outcome train/test: 0.9188 / 0.7782
  R2 Treatment train/test: 0.9094 / 0.6818


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.198218,0.168780,0.240419,0.192626
Boosting (Tabular),0.592756,0.522182,0.562144,0.404502
OLS (Tabular + PCA),0.658984,0.575474,0.619038,0.510152
Boosting (Tabular + PCA),0.902234,0.771662,0.879742,0.690785
OLS (Tabular + Similarities),0.676718,0.587811,0.628433,0.514226
Boosting (Tabular + Similarities),0.882863,0.756810,0.842498,0.651803
OLS (Tabular + Embeddings),0.830711,0.708981,0.789709,0.638545
Boosting (Tabular + Embeddings),0.918809,0.778197,0.909382,0.681802


## ⑬ Summary — Level Models

In [16]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.198218,0.168780,0.240419,0.192626
Boosting (Tabular),0.592756,0.522182,0.562144,0.404502
OLS (Tabular + PCA),0.658984,0.575474,0.619038,0.510152
Boosting (Tabular + PCA),0.902234,0.771662,0.879742,0.690785
OLS (Tabular + Similarities),0.676718,0.587811,0.628433,0.514226
Boosting (Tabular + Similarities),0.882863,0.756810,0.842498,0.651803
OLS (Tabular + Embeddings),0.830711,0.708981,0.789709,0.638545
Boosting (Tabular + Embeddings),0.918809,0.778197,0.909382,0.681802
Deep Time Independent,0.822300,0.716300,0.775600,0.639000
Deep Time Dependent,0.902100,0.891300,0.779600,0.644800


## ⑭ Tabular Models — Diff

In [17]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.1634 / 0.1750
  R2 Treatment diff train/test: 0.0185 / 0.0149
  Boosting
  R2 Outcome diff train/test: 0.3649 / 0.2991
  R2 Treatment diff train/test: 0.1234 / 0.0517

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.1649 / 0.1763
  R2 Treatment diff train/test: 0.0187 / 0.0153
  Boosting
  R2 Outcome diff train/test: 0.4674 / 0.3546
  R2 Treatment diff train/test: 0.1823 / 0.0553

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.1654 / 0.1756
  R2 Treatment diff train/test: 0.0187 / 0.0153
  Boosting
  R2 Outcome diff train/test: 0.4293 / 0.3230
  R2 Treatment diff train/test: 0.1730 / 0.0442

 Feature specification: x_emb
  OLS
  R2 Outcome diff train/test: 0.1847 / 0.1805
  R2 Treatment diff train/test: 0.0333 / 0.0125
  Boosting
  R2 Outcome diff train/test: 0.5519 / 0.3670
  R2 Treatment diff train/test: 0.2678 / 0.0316


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.163428,0.174993,0.018508,0.014902
Boosting (Tabular),0.364878,0.299111,0.123402,0.051698
OLS (Tabular + PCA),0.164942,0.176343,0.018733,0.015314
Boosting (Tabular + PCA),0.467368,0.354554,0.182314,0.055269
OLS (Tabular + Similarities),0.165423,0.175617,0.018694,0.015259
Boosting (Tabular + Similarities),0.429274,0.322956,0.172973,0.044163
OLS (Tabular + Embeddings),0.184745,0.180527,0.033297,0.012546
Boosting (Tabular + Embeddings),0.551877,0.367004,0.267844,0.031569


## ⑮ Summary — Diff Models

In [18]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.163428,0.174993,0.018508,0.014902
Boosting (Tabular),0.364878,0.299111,0.123402,0.051698
OLS (Tabular + PCA),0.164942,0.176343,0.018733,0.015314
Boosting (Tabular + PCA),0.467368,0.354554,0.182314,0.055269
OLS (Tabular + Similarities),0.165423,0.175617,0.018694,0.015259
Boosting (Tabular + Similarities),0.429274,0.322956,0.172973,0.044163
OLS (Tabular + Embeddings),0.184745,0.180527,0.033297,0.012546
Boosting (Tabular + Embeddings),0.551877,0.367004,0.267844,0.031569
Deep Time Independent,0.110429,0.073140,-0.019514,-0.017426
Deep Time Dependent,0.069633,0.063880,-0.009364,-0.009239


## ⑯ Final Summary (% format)

R² multiplied by 100 — matches paper Table 2 format.
Compare with 03_2 (txt only) to see the gain from adding image embeddings.

In [19]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          16.88      19.26
Boosting (Tabular)                     52.22      40.45
OLS (Tabular + PCA)                    57.55      51.02
Boosting (Tabular + PCA)               77.17      69.08
OLS (Tabular + Similarities)           58.78      51.42
Boosting (Tabular + Similarities)      75.68      65.18
OLS (Tabular + Embeddings)             70.90      63.85
Boosting (Tabular + Embeddings)        77.82      68.18
Deep Time Independent                  71.63      63.90
Deep Time Dependent                    89.13      64.48

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          17.50       1.49
Boosting (Tabular)                     29.91       5.17
OLS (Tabular + PCA)                    17.63       1.53
Boosting (Tabular + PCA)               35.46       5.53
OLS (Tabular + Similarities)      